# Day 3: Testing & CI/CD for ML

**MLOps in Practice Workshop**

---

## Learning Objectives

1. Write unit and integration tests for ML code with pytest
2. Validate data quality with Great Expectations
3. Design CI/CD pipelines with GitHub Actions
4. Understand the components of a production ML CI/CD pipeline

---

## 1. Testing ML Code with Pytest

ML code needs the same testing discipline as any software, plus additional concerns:

| Test Type | What It Checks | Example |
|-----------|---------------|---------|
| **Unit tests** | Individual functions work correctly | `test_preprocess_scales_features()` |
| **Integration tests** | Components work together | `test_pipeline_train_and_predict()` |
| **Data tests** | Input data meets expectations | `test_no_null_values()` |
| **Model tests** | Model quality meets thresholds | `test_accuracy_above_threshold()` |
| **Regression tests** | New model is not worse than old | `test_new_model_not_degraded()` |

In [ ]:
# Install dependencies
# !pip install pytest scikit-learn pandas numpy great-expectations

### 1.1 The Code Under Test

First, let us define the production code we want to test.

In [ ]:
# src/data/make_dataset.py -- Data loading and preprocessing

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler


def load_data(path: str) -> pd.DataFrame:
    """Load a CSV dataset and perform basic validation."""
    df = pd.read_csv(path)
    if df.empty:
        raise ValueError("Dataset is empty")
    return df


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Remove duplicates and handle missing values."""
    df = df.drop_duplicates()
    # Fill numeric columns with median
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
    return df


def extract_features(df: pd.DataFrame, feature_cols: list) -> np.ndarray:
    """Extract and scale feature columns."""
    if not all(col in df.columns for col in feature_cols):
        missing = set(feature_cols) - set(df.columns)
        raise KeyError(f"Missing columns: {missing}")
    X = df[feature_cols].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, scaler


print("Data module defined.")

In [ ]:
# src/models/train.py -- Model training

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score


def train_model(
    X_train: np.ndarray,
    y_train: np.ndarray,
    n_estimators: int = 100,
    max_depth: int = None,
    random_state: int = 42,
) -> RandomForestClassifier:
    """Train a Random Forest classifier."""
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state,
    )
    model.fit(X_train, y_train)
    return model


def evaluate_model(model, X_test, y_test) -> dict:
    """Evaluate model and return metrics."""
    y_pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted"),
    }


print("Model module defined.")

### 1.2 Writing Pytest Tests

Here are real test functions you would put in `tests/test_data.py` and `tests/test_model.py`.

In [ ]:
# tests/test_data.py

import pytest
import pandas as pd
import numpy as np


# ---- Fixtures ----

@pytest.fixture
def sample_df():
    """Create a sample DataFrame for testing."""
    return pd.DataFrame({
        "feature_a": [1.0, 2.0, 3.0, 4.0, 5.0, 5.0],
        "feature_b": [10.0, 20.0, np.nan, 40.0, 50.0, 50.0],
        "label": [0, 1, 0, 1, 0, 0],
    })


@pytest.fixture
def empty_df():
    return pd.DataFrame()


# ---- Tests for clean_data ----

def test_clean_data_removes_duplicates(sample_df):
    """clean_data should remove duplicate rows."""
    result = clean_data(sample_df)
    assert len(result) == 5  # one duplicate removed


def test_clean_data_fills_missing_values(sample_df):
    """clean_data should fill NaN with median."""
    result = clean_data(sample_df)
    assert result["feature_b"].isna().sum() == 0


def test_clean_data_preserves_shape(sample_df):
    """clean_data should not add or remove columns."""
    result = clean_data(sample_df)
    assert result.shape[1] == sample_df.shape[1]


# ---- Tests for extract_features ----

def test_extract_features_shape(sample_df):
    """extract_features should return correct shape."""
    cleaned = clean_data(sample_df)
    X, scaler = extract_features(cleaned, ["feature_a", "feature_b"])
    assert X.shape == (5, 2)


def test_extract_features_scaled(sample_df):
    """Scaled features should have mean near 0 and std near 1."""
    cleaned = clean_data(sample_df)
    X, scaler = extract_features(cleaned, ["feature_a", "feature_b"])
    assert np.abs(X.mean(axis=0)).max() < 1e-10
    assert np.abs(X.std(axis=0, ddof=0) - 1.0).max() < 1e-10


def test_extract_features_missing_column(sample_df):
    """extract_features should raise KeyError for missing columns."""
    with pytest.raises(KeyError, match="Missing columns"):
        extract_features(sample_df, ["feature_a", "nonexistent"])


print("Data tests defined. Run with: pytest tests/test_data.py -v")

In [ ]:
# tests/test_model.py

import pytest
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split


@pytest.fixture
def iris_data():
    """Prepare iris train/test split."""
    iris = load_iris()
    X_train, X_test, y_train, y_test = train_test_split(
        iris.data, iris.target, test_size=0.2, random_state=42
    )
    return X_train, X_test, y_train, y_test


@pytest.fixture
def trained_model(iris_data):
    """Train a model for use in tests."""
    X_train, _, y_train, _ = iris_data
    return train_model(X_train, y_train, n_estimators=50)


# ---- Model Training Tests ----

def test_train_model_returns_fitted_model(iris_data):
    """train_model should return a fitted RandomForestClassifier."""
    X_train, _, y_train, _ = iris_data
    model = train_model(X_train, y_train)
    assert hasattr(model, "predict")
    assert hasattr(model, "predict_proba")
    # Check that the model is fitted (has n_classes_)
    assert hasattr(model, "n_classes_")


def test_train_model_prediction_shape(iris_data, trained_model):
    """Predictions should have the same length as test set."""
    _, X_test, _, y_test = iris_data
    y_pred = trained_model.predict(X_test)
    assert len(y_pred) == len(y_test)


def test_train_model_predicts_valid_classes(iris_data, trained_model):
    """All predictions should be valid class labels."""
    _, X_test, _, _ = iris_data
    y_pred = trained_model.predict(X_test)
    assert set(y_pred).issubset({0, 1, 2})


# ---- Model Quality Tests ----

def test_accuracy_above_threshold(iris_data, trained_model):
    """Model accuracy should be above minimum threshold."""
    _, X_test, _, y_test = iris_data
    metrics = evaluate_model(trained_model, X_test, y_test)
    assert metrics["accuracy"] > 0.85, (
        f"Accuracy {metrics['accuracy']:.4f} is below threshold 0.85"
    )


def test_f1_above_threshold(iris_data, trained_model):
    """Model F1 score should be above minimum threshold."""
    _, X_test, _, y_test = iris_data
    metrics = evaluate_model(trained_model, X_test, y_test)
    assert metrics["f1_weighted"] > 0.85


def test_model_deterministic(iris_data):
    """Same random_state should produce identical models."""
    X_train, X_test, y_train, _ = iris_data
    model_a = train_model(X_train, y_train, random_state=42)
    model_b = train_model(X_train, y_train, random_state=42)
    pred_a = model_a.predict(X_test)
    pred_b = model_b.predict(X_test)
    np.testing.assert_array_equal(pred_a, pred_b)


print("Model tests defined. Run with: pytest tests/test_model.py -v")

In [ ]:
# Run the tests directly in the notebook using pytest's programmatic API
# In a real project, you would run: pytest tests/ -v

# Simulate running tests
print("=" * 60)
print("Running data tests...")
print("=" * 60)

sample_data = pd.DataFrame({
    "feature_a": [1.0, 2.0, 3.0, 4.0, 5.0, 5.0],
    "feature_b": [10.0, 20.0, np.nan, 40.0, 50.0, 50.0],
    "label": [0, 1, 0, 1, 0, 0],
})

# Test: clean_data removes duplicates
cleaned = clean_data(sample_data)
assert len(cleaned) == 5, "FAIL: duplicates not removed"
print("PASS: test_clean_data_removes_duplicates")

# Test: clean_data fills NaN
assert cleaned["feature_b"].isna().sum() == 0, "FAIL: NaN not filled"
print("PASS: test_clean_data_fills_missing_values")

# Test: extract_features shape
X, scaler = extract_features(cleaned, ["feature_a", "feature_b"])
assert X.shape == (5, 2), "FAIL: wrong shape"
print("PASS: test_extract_features_shape")

# Test: extract_features scaling
assert np.abs(X.mean(axis=0)).max() < 1e-10, "FAIL: mean not zero"
print("PASS: test_extract_features_scaled")

# Test: missing column raises error
try:
    extract_features(sample_data, ["feature_a", "nonexistent"])
    print("FAIL: expected KeyError")
except KeyError:
    print("PASS: test_extract_features_missing_column")

print("\n" + "=" * 60)
print("Running model tests...")
print("=" * 60)

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

model = train_model(X_train, y_train, n_estimators=50)

assert hasattr(model, "predict"), "FAIL: model not fitted"
print("PASS: test_train_model_returns_fitted_model")

y_pred = model.predict(X_test)
assert len(y_pred) == len(y_test), "FAIL: wrong prediction length"
print("PASS: test_train_model_prediction_shape")

assert set(y_pred).issubset({0, 1, 2}), "FAIL: invalid classes"
print("PASS: test_train_model_predicts_valid_classes")

metrics = evaluate_model(model, X_test, y_test)
assert metrics["accuracy"] > 0.85, f"FAIL: accuracy={metrics['accuracy']}"
print(f"PASS: test_accuracy_above_threshold (accuracy={metrics['accuracy']:.4f})")

assert metrics["f1_weighted"] > 0.85, f"FAIL: f1={metrics['f1_weighted']}"
print(f"PASS: test_f1_above_threshold (f1={metrics['f1_weighted']:.4f})")

print("\nAll tests passed!")

---

## 2. Data Validation with Great Expectations

Great Expectations defines "expectations" -- assertions about your data that are checked automatically.

In [ ]:
# Great Expectations example -- defining expectation suites

import pandas as pd
import numpy as np

# Sample training data
train_df = pd.DataFrame({
    "age": np.random.randint(18, 80, size=1000),
    "income": np.random.normal(50000, 15000, size=1000).clip(0),
    "credit_score": np.random.randint(300, 850, size=1000),
    "loan_amount": np.random.normal(20000, 8000, size=1000).clip(0),
    "default": np.random.binomial(1, 0.15, size=1000),
})

print(f"Training data shape: {train_df.shape}")
print(train_df.describe().round(2))

In [ ]:
# Define expectation suite (what we expect from valid data)
# In Great Expectations, this would be saved as a JSON suite.
# Here we show the logic as plain validation functions.

def validate_loan_data(df: pd.DataFrame) -> dict:
    """
    Validate a loan dataset against expectations.
    Returns a dict with pass/fail results.
    """
    results = {}

    # Expectation 1: Required columns exist
    required_cols = {"age", "income", "credit_score", "loan_amount", "default"}
    results["has_required_columns"] = required_cols.issubset(set(df.columns))

    # Expectation 2: No null values in critical columns
    results["no_nulls_age"] = df["age"].isna().sum() == 0
    results["no_nulls_income"] = df["income"].isna().sum() == 0
    results["no_nulls_credit_score"] = df["credit_score"].isna().sum() == 0

    # Expectation 3: Values in valid ranges
    results["age_in_range"] = df["age"].between(0, 120).all()
    results["credit_score_in_range"] = df["credit_score"].between(300, 850).all()
    results["income_non_negative"] = (df["income"] >= 0).all()
    results["loan_amount_non_negative"] = (df["loan_amount"] >= 0).all()

    # Expectation 4: Target column is binary
    results["default_is_binary"] = set(df["default"].unique()).issubset({0, 1})

    # Expectation 5: Minimum number of rows
    results["min_row_count"] = len(df) >= 100

    # Expectation 6: Target class balance (not too skewed)
    default_rate = df["default"].mean()
    results["target_balance"] = 0.01 < default_rate < 0.99

    return results


# Validate training data
validation_results = validate_loan_data(train_df)

print("Validation Results:")
print("-" * 40)
all_passed = True
for expectation, passed in validation_results.items():
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"  {status}: {expectation}")

print(f"\nOverall: {'ALL PASSED' if all_passed else 'SOME FAILED'}")

In [ ]:
# Great Expectations -- what the actual API looks like
# (requires: pip install great-expectations)

ge_example = '''
import great_expectations as gx

# Create a context
context = gx.get_context()

# Create a data source and get a batch
validator = context.sources.pandas_default.read_dataframe(train_df)

# Define expectations
validator.expect_column_values_to_not_be_null("age")
validator.expect_column_values_to_not_be_null("income")
validator.expect_column_values_to_be_between("age", min_value=0, max_value=120)
validator.expect_column_values_to_be_between("credit_score", min_value=300, max_value=850)
validator.expect_column_values_to_be_in_set("default", [0, 1])
validator.expect_table_row_count_to_be_between(min_value=100)

# Run validation
results = validator.validate()
print(f"Success: {results.success}")
'''

print("Great Expectations API example:")
print(ge_example)

---

## 3. GitHub Actions Workflow

GitHub Actions automates testing, building, and deployment on every push or pull request.

### 3.1 ML CI/CD Workflow

```yaml
# .github/workflows/ml-pipeline.yml
name: ML Pipeline CI/CD

on:
  push:
    branches: [main, develop]
  pull_request:
    branches: [main]

env:
  PYTHON_VERSION: "3.10"
  MLFLOW_TRACKING_URI: sqlite:///mlflow.db

jobs:
  lint-and-format:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}

      - name: Install linters
        run: pip install ruff black

      - name: Run linter
        run: ruff check src/ tests/

      - name: Check formatting
        run: black --check src/ tests/

  test:
    needs: lint-and-format
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}

      - name: Cache pip dependencies
        uses: actions/cache@v4
        with:
          path: ~/.cache/pip
          key: ${{ runner.os }}-pip-${{ hashFiles('requirements.txt') }}

      - name: Install dependencies
        run: |
          pip install -r requirements.txt
          pip install pytest pytest-cov

      - name: Run unit tests
        run: pytest tests/ -v --cov=src --cov-report=xml

      - name: Upload coverage
        uses: codecov/codecov-action@v4
        with:
          file: coverage.xml

  train-and-evaluate:
    needs: test
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Pull data with DVC
        run: dvc pull

      - name: Train model
        run: python src/models/train.py

      - name: Evaluate model
        run: python src/evaluation/evaluate.py

      - name: Check model quality gate
        run: |
          python -c "
          import json
          with open('reports/metrics.json') as f:
              metrics = json.load(f)
          assert metrics['accuracy'] > 0.85, f'Accuracy {metrics[\"accuracy\"]} below threshold'
          print(f'Quality gate passed: accuracy={metrics[\"accuracy\"]:.4f}')
          "

      - name: Upload model artifact
        uses: actions/upload-artifact@v4
        with:
          name: trained-model
          path: models/

  build-and-push:
    needs: train-and-evaluate
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Download model artifact
        uses: actions/download-artifact@v4
        with:
          name: trained-model
          path: models/

      - name: Build Docker image
        run: docker build -t ml-api:${{ github.sha }} .

      - name: Push to registry
        run: |
          echo "${{ secrets.DOCKER_PASSWORD }}" | docker login -u ${{ secrets.DOCKER_USERNAME }} --password-stdin
          docker tag ml-api:${{ github.sha }} myregistry/ml-api:${{ github.sha }}
          docker tag ml-api:${{ github.sha }} myregistry/ml-api:latest
          docker push myregistry/ml-api:${{ github.sha }}
          docker push myregistry/ml-api:latest
```

### 3.2 CI/CD Pipeline Design

A well-designed ML CI/CD pipeline has these stages:

```
+----------+     +---------+     +----------+     +---------+     +--------+
| Lint &   | --> | Unit    | --> | Train &  | --> | Build   | --> | Deploy |
| Format   |     | Tests   |     | Evaluate |     | Docker  |     |        |
+----------+     +---------+     +----------+     +---------+     +--------+
                                      |
                                      v
                              Quality Gate Check
                            (accuracy > threshold?)
```

### Key Design Principles

1. **Fail fast** -- run linting and unit tests first (they are quick)
2. **Quality gates** -- never deploy a model that does not meet minimum quality
3. **Artifact versioning** -- tag Docker images with Git SHA for traceability
4. **Data validation** -- validate input data before training
5. **Separate concerns** -- training pipeline vs. deployment pipeline

---

## 4. Lab: Write Tests for Data Pipeline and Model

### Objective

Write a complete test suite for an ML pipeline that:
1. Loads and validates data
2. Preprocesses features
3. Trains a model
4. Evaluates model quality

### The Pipeline Code

In [ ]:
# The pipeline to test

import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score


class MLPipeline:
    """A simple ML pipeline for binary classification."""

    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.scaler = StandardScaler()
        self.model = None

    def validate_data(self, df: pd.DataFrame, required_cols: list) -> bool:
        """Check that the DataFrame has required columns and no nulls."""
        if not all(col in df.columns for col in required_cols):
            return False
        if df[required_cols].isnull().any().any():
            return False
        if len(df) < 10:
            return False
        return True

    def preprocess(self, X_train, X_test):
        """Scale features."""
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        return X_train_scaled, X_test_scaled

    def train(self, X_train, y_train):
        """Train the model."""
        self.model = GradientBoostingClassifier(
            n_estimators=self.n_estimators,
            learning_rate=self.learning_rate,
            max_depth=self.max_depth,
            random_state=42,
        )
        self.model.fit(X_train, y_train)
        return self.model

    def evaluate(self, X_test, y_test) -> dict:
        """Evaluate and return metrics."""
        if self.model is None:
            raise RuntimeError("Model not trained yet")
        y_pred = self.model.predict(X_test)
        return {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, average="weighted"),
            "recall": recall_score(y_test, y_pred, average="weighted"),
        }


print("MLPipeline class defined.")

### Your Task: Write Tests

Write tests for each method of `MLPipeline`. Cover:

1. `validate_data` -- valid data, missing columns, null values, too few rows
2. `preprocess` -- output shape, scaling properties
3. `train` -- model is fitted, correct type
4. `evaluate` -- metrics are in valid range, error if model not trained

In [ ]:
# ============================================================
# LAB: Write your tests here
# ============================================================

# Generate synthetic data for testing
X_synth, y_synth = make_classification(
    n_samples=200, n_features=5, n_classes=2, random_state=42
)
feature_names = [f"feat_{i}" for i in range(5)]
synth_df = pd.DataFrame(X_synth, columns=feature_names)
synth_df["target"] = y_synth

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_synth, y_synth, test_size=0.2, random_state=42
)

pipeline = MLPipeline(n_estimators=50)

# --- Test validate_data ---
print("Testing validate_data...")

# Test: valid data passes
assert pipeline.validate_data(synth_df, feature_names) == True
print("  PASS: valid data accepted")

# Test: missing columns fails
assert pipeline.validate_data(synth_df, ["nonexistent"]) == False
print("  PASS: missing columns rejected")

# Test: null values fail
df_with_nulls = synth_df.copy()
df_with_nulls.loc[0, "feat_0"] = np.nan
assert pipeline.validate_data(df_with_nulls, feature_names) == False
print("  PASS: null values rejected")

# Test: too few rows fails
assert pipeline.validate_data(synth_df.head(5), feature_names) == False
print("  PASS: too few rows rejected")

# --- Test preprocess ---
print("\nTesting preprocess...")

X_tr_scaled, X_te_scaled = pipeline.preprocess(X_train_s, X_test_s)
assert X_tr_scaled.shape == X_train_s.shape
print("  PASS: output shape matches input")

assert np.abs(X_tr_scaled.mean(axis=0)).max() < 1e-10
print("  PASS: training data has zero mean")

# --- Test train ---
print("\nTesting train...")

model = pipeline.train(X_tr_scaled, y_train_s)
assert hasattr(model, "predict")
print("  PASS: model is fitted")

assert isinstance(model, GradientBoostingClassifier)
print("  PASS: correct model type")

# --- Test evaluate ---
print("\nTesting evaluate...")

metrics = pipeline.evaluate(X_te_scaled, y_test_s)
assert 0 <= metrics["accuracy"] <= 1
assert 0 <= metrics["precision"] <= 1
assert 0 <= metrics["recall"] <= 1
print(f"  PASS: metrics in valid range (accuracy={metrics['accuracy']:.4f})")

# Test: evaluate before training raises error
untrained = MLPipeline()
try:
    untrained.evaluate(X_te_scaled, y_test_s)
    print("  FAIL: expected RuntimeError")
except RuntimeError:
    print("  PASS: RuntimeError raised for untrained model")

print("\nAll lab tests passed!")

---

## Summary

| Topic | Key Takeaway |
|-------|-------------|
| **pytest** | Use fixtures, parametrize, and clear assertions; test data, models, and integration |
| **Great Expectations** | Define data contracts; validate data quality before training |
| **GitHub Actions** | Automate linting, testing, training, and deployment |
| **Quality gates** | Never deploy a model below minimum accuracy/F1 threshold |
| **CI/CD design** | Fail fast, version artifacts, separate training from deployment |

### Next Session

**Day 4: Monitoring & Production** -- We will detect data drift and design monitoring systems.